# II-2 Tailscale：経路ごとの往復時間（RTT）を比べる（ノートブック版）

教科書の II-2 のうち、**Pi と Tailscale がなくても試せる解析**（II-2-6 の `rtt_compare.py`）を、合成の RTT のデータで試せるようにしたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac で全部で約15秒）。

| 節 | 試すこと | 教科書の手順 | 使うファイル |
|---|---|---|---|
| 1 | 合成の RTT のデータを作る（3つの経路の測定の代わり） | 手順の「RTT を測る」 | `synth_rtt.py`（ノートブック版で追加） |
| 2 | 経路ごとの中央値・95%点・欠損率（Wilson）、2通りの区間、中央値の差、図 | 「比べる」 | `rtt_compare.py` |
| 3 | 答えのわかったデータで、2通りの区間が真の中央値を含む割合（被覆率）を数える | II-2-3「統計の誤用」 | `rtt_compare.py` の関数（ノートブックで追加の実験） |
| 4 | `rtt_measure.py` が ping の表示から RTT を読む部分（Mac なら 127.0.0.1 に実際に ping） | 「RTT を測る」 | `rtt_measure.py` |

- 次のセルたちが、II-2-6 の `rtt_measure.py`・`rtt_compare.py` と、合成データを作る `synth_rtt.py` を、このノートブックの隣に `%%writefile` で書き出します。本文のコードで変えたのは、図の日本語フォントの候補を足したこと（`plt.rcParams["font.family"]` の行）だけです。
- `synth_rtt.py` は本文にはありません。経路ごとに、ふだんの RTT（$\log$ RTT が AR(1)：となりどうしが似ている）に、ときどき数秒続く「混雑」（数十 ms の上乗せ、届かないことが増える）を足し、`rtt_measure.py` と同じ列（`label, when, seq, rtt_ms`、届かなかったら空欄）の `rtt_synth.csv` を作ります。経路は本文と同じ `lan`・`ts_home`・`ts_outside` で、ふだんの中央値は 4.0・4.6・45 ms にしました（本文 II-2-7 の目安：lan と ts_home は数 ms で差は 1 ms 前後以内、ts_outside は数十 ms）。**実測の値ではありません**。
- 各節では、ターミナルの `uv run python ○○.py …` の代わりに、`run("○○.py …")` で同じ引数のまま実行します。`ok()` で本文の手順の「確かめること」を確かめます。
- **入れていないもの（Pi・Tailscale が要る部分）**：Tailscale を入れてログインすること、MagicDNS・ACL・SSH の設定、外出先（テザリング）からの測定。`rtt_measure.py` は 4 節で ping の表示を読む部分だけを試します。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・pandas・matplotlib・statsmodels が入っている。版は要確認）
import os
for _v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ.setdefault(_v, "1")   # 小さい行列の計算だけなので1スレッドにする（CPU が混んでいると、多スレッドはかえって遅くなる）
import importlib, subprocess, sys
for mod, pkg in [("statsmodels", "statsmodels"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import statsmodels, numpy, pandas, matplotlib
print("OK", "numpy", numpy.__version__, "pandas", pandas.__version__, "statsmodels", statsmodels.__version__, "matplotlib", matplotlib.__version__)
print("（ノートブックは numpy 2.5.5・pandas 3.1.1・statsmodels 0.15.0・matplotlib 3.11.2 で確かめた）")

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

次の3つのセルで、ファイルを書き出します。最後の `synth_rtt.py` は、ノートブック版で追加した合成データのプログラムです。

In [ ]:
%%writefile rtt_measure.py
#!/usr/bin/env python3
"""II-2：ping で往復時間（RTT）を測り、経路の名前（ラベル）をつけて CSV に追記する

使い方（Mac で）: cd ~/home-lab/partII/ii2
  uv run python rtt_measure.py --host labpi.local --label lan --count 300
  uv run python rtt_measure.py --host labpi --label ts_home --count 300        # Tailscale（MagicDNS の名前）
  uv run python rtt_measure.py --host labpi --label ts_outside --count 300     # 外出先・テザリングなど
  --interval : 送る間隔 [s]（既定 0.2。macOS では 0.1 未満は管理者権限が必要）
結果: rtt.csv に label, when, seq, rtt_ms（届かなかったら空欄）を追記する
"""
import argparse
import csv
import datetime
import pathlib
import re
import subprocess
import sys

LINE = re.compile(r"icmp_seq=(\d+).*?time=([\d.]+)\s*ms")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--host", required=True)
    ap.add_argument("--label", required=True)
    ap.add_argument("--count", type=int, default=300)
    ap.add_argument("--interval", type=float, default=0.2)
    ap.add_argument("--out", default="rtt.csv")
    a = ap.parse_args()
    cmd = ["ping", "-c", str(a.count), "-i", str(a.interval), a.host]
    print("実行:", " ".join(cmd), f"（約 {a.count * a.interval:.0f} 秒）")
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=a.count * a.interval + 60).stdout
    except subprocess.TimeoutExpired:
        sys.exit("ping が終わりませんでした")
    got = {int(m.group(1)): float(m.group(2)) for m in map(LINE.search, out.splitlines()) if m}
    if not got:
        sys.exit("1つも返事がありませんでした。名前の解決（tailscale status）や ACL を確かめてください\n" + out[-500:])
    first = 0                                            # macOS の ping は icmp_seq を 0 から数える（この道具は Mac 用）
    when = datetime.datetime.now().isoformat(timespec="seconds")
    path = pathlib.Path(a.out)
    new = not path.exists()
    with open(path, "a", newline="") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["label", "when", "seq", "rtt_ms"])
        for s in range(first, first + a.count):
            w.writerow([a.label, when, s - first, got.get(s, "")])
    lost = a.count - len(got)
    r = sorted(got.values())
    print(f"{a.label}: 受信 {len(got)}/{a.count}（欠損 {lost}）、中央値 {r[len(r)//2]:.2f} ms、最大 {r[-1]:.2f} ms → {path} に追記")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile rtt_compare.py
#!/usr/bin/env python3
"""II-2：経路ごとの往復時間（RTT）を比べる

使い方: cd ~/home-lab/partII/ii2 && uv run python rtt_compare.py rtt.csv

やること
  1. 経路（label）ごとに、中央値・95%点・最大・欠損率（Wilson 区間）
  2. 中央値の 95% 区間を2通りで出す
     【誤り】1回1回の ping を独立とみなしたブートストラップ
     【まし】となりどうしが似ていることを考えた「移動ブロック・ブートストラップ」（ブロックの長さ 20）
  3. 経路どうしの中央値の差（ブロック・ブートストラップ）と、経験分布関数の図
RTT は裾が重い（ときどき大きな値が出る）ので、平均より中央値や分位点で比べる。
"""
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
RNG = np.random.default_rng(0)
NB = 3000
BLOCK = 20


def iid_boot(x):
    return np.array([np.median(RNG.choice(x, len(x))) for _ in range(NB)])


def block_boot(x, b=BLOCK):
    n = len(x)
    k = int(np.ceil(n / b))
    out = []
    for _ in range(NB):
        starts = RNG.integers(0, n - b + 1, k)
        out.append(np.median(np.concatenate([x[s:s + b] for s in starts])[:n]))
    return np.array(out)


def main(path):
    d = pd.read_csv(path)
    labels = list(dict.fromkeys(d["label"]))
    boots = {}
    print(f"{'経路':12s} {'受信':>9s} {'欠損率[%]':>10s} {'(Wilson)':>14s} {'中央値':>7s} {'95%点':>7s} {'最大':>7s} {'ρ(1)':>6s}")
    for lb in labels:
        g = d[d["label"] == lb]
        x = g["rtt_ms"].dropna().to_numpy()
        n, k = len(g), int(g["rtt_ms"].isna().sum())
        lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson")
        r1 = np.corrcoef(x[:-1], x[1:])[0, 1] if len(x) > 3 else np.nan
        print(f"{lb:12s} {n-k:4d}/{n:<4d} {k/n*100:10.2f} {f'({lo*100:.1f}〜{hi*100:.1f})':>14s} "
              f"{np.median(x):7.2f} {np.quantile(x, 0.95):7.2f} {x.max():7.1f} {r1:6.2f}")
        bi, bb = iid_boot(x), block_boot(x)
        boots[lb] = bb
        print(f"{'':12s} 中央値の95%区間：独立とみなす {np.percentile(bi, 2.5):.2f}〜{np.percentile(bi, 97.5):.2f} ms、"
              f"ブロック {np.percentile(bb, 2.5):.2f}〜{np.percentile(bb, 97.5):.2f} ms")
    print("・ρ(1) はとなりあう ping どうしの相関。大きいと、独立とみなした区間は狭すぎる（混雑は続けて起こる）。")
    print("・ブロック・ブートストラップも「1回の測定の時間帯」についての区間。別の日・別の時間帯のばらつきは含まない。")
    if len(labels) >= 2:
        print("\n[中央値の差]（ブロック・ブートストラップ、95% 区間）")
        base = labels[0]
        for lb in labels[1:]:
            diff = boots[lb] - boots[base]
            print(f"  {lb} − {base}：{np.median(diff):+.2f} ms（{np.percentile(diff, 2.5):+.2f}〜{np.percentile(diff, 97.5):+.2f}）")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
    for lb in labels:
        x = np.sort(d[d["label"] == lb]["rtt_ms"].dropna().to_numpy())
        ax[0].step(x, np.arange(1, len(x) + 1) / len(x), where="post", label=lb)
        ax[1].plot(d[d["label"] == lb]["rtt_ms"].to_numpy(), ".", ms=2, label=lb)
    ax[0].set(xscale="log", xlabel="RTT [ms]（対数目盛）", ylabel="累積割合", title="経験分布関数")
    ax[1].set(yscale="log", xlabel="何回目か", ylabel="RTT [ms]", title="時間の順に並べた RTT（大きい値が続けて出るか）")
    for x in ax:
        x.grid(True, which="both", alpha=0.3)
        x.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("rtt.png", dpi=130)
    print("\nrtt.png を保存しました")


if __name__ == "__main__":
    if len(sys.argv) < 2:
        sys.exit("使い方: uv run python rtt_compare.py rtt.csv")
    main(sys.argv[1])

In [ ]:
%%writefile synth_rtt.py
#!/usr/bin/env python3
"""II-2 のノートブック版で追加：rtt_measure.py の rtt.csv と同じ形の、合成の往復時間（RTT）を作る

ふだんの RTT：log(RTT) が AR(1)（となりどうしが似ている。0.2 秒おきの ping の自己相関のまね）
混雑：ときどき「混雑」の状態に入り、数秒続く（その間は数十 ms 足される）。混雑の間は届かないことも多い
使い方: python synth_rtt.py --out rtt_synth.csv [--count 300] [--seed 1]
"""
import argparse
import csv
import json

import numpy as np

# 経路ごとの設定：ふだんの中央値 [ms]、log のばらつき、AR(1) の係数、混雑に入る確率／続く確率、混雑の上乗せ [ms]、欠損の確率（ふだん／混雑）
ROUTES = {
    "lan":        dict(med=4.0,  sig=0.25, phi=0.7,  p_in=0.010, p_stay=0.90, add=30.0, loss=(0.002, 0.05)),
    "ts_home":    dict(med=4.6,  sig=0.25, phi=0.7,  p_in=0.010, p_stay=0.90, add=30.0, loss=(0.002, 0.05)),
    "ts_outside": dict(med=45.0, sig=0.30, phi=0.85, p_in=0.020, p_stay=0.93, add=80.0, loss=(0.005, 0.10)),
}


def simulate(route, n, rng):
    c = ROUTES[route]
    e = rng.normal(0, c["sig"])
    busy = False
    out = []
    for _ in range(n):
        e = c["phi"] * e + c["sig"] * np.sqrt(1 - c["phi"] ** 2) * rng.normal()
        busy = rng.random() < (c["p_stay"] if busy else c["p_in"])
        x = c["med"] * np.exp(e) + (c["add"] * rng.lognormal(0, 0.5) if busy else 0.0)
        lost = rng.random() < c["loss"][busy]
        out.append(np.nan if lost else x)
    return np.array(out)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="rtt_synth.csv")
    ap.add_argument("--count", type=int, default=300)
    ap.add_argument("--seed", type=int, default=1)
    a = ap.parse_args()
    rng = np.random.default_rng(a.seed)
    with open(a.out, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["label", "when", "seq", "rtt_ms"])
        for k, lb in enumerate(ROUTES):
            x = simulate(lb, a.count, rng)
            when = f"2026-11-27T21:{10 + 3 * k:02d}:00"
            for s, v in enumerate(x):
                w.writerow([lb, when, s, "" if np.isnan(v) else f"{v:.3f}"])
    # 真の中央値（とても長い列で求める。確かめだけに使う）
    truth = {lb: float(np.nanmedian(simulate(lb, 200_000, np.random.default_rng(99)))) for lb in ROUTES}
    json.dump(truth, open(a.out.replace(".csv", "_truth.json"), "w"), indent=1)
    print(f"{a.out} に {len(ROUTES)} 経路 × {a.count} 回を書きました。真の中央値 [ms]：",
          {k: round(v, 3) for k, v in truth.items()})


if __name__ == "__main__":
    main()

In [ ]:
import json, os, re, shlex, shutil
import numpy as np
import pandas as pd
from IPython.display import Image, display


def run(args):
    """ターミナルの「uv run python ○○.py …」の代わり。表示を出し、あとで確かめられるように文字列で返す"""
    p = subprocess.run([sys.executable] + shlex.split(args), capture_output=True, text=True)
    print(p.stdout, end="")
    keep = [l for l in p.stderr.splitlines() if "findfont" not in l and "font cache" not in l and l.strip()]
    if keep:
        print("\n".join(keep), file=sys.stderr)
    assert p.returncode == 0, "★実行に失敗：" + args
    return p.stdout


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, text, i=1):
    m = re.search(pattern, text)
    assert m, "表示に見つからない：" + pattern
    return float(m.group(i).replace("−", "-"))


print("準備ができました")

## 1. 合成の RTT のデータを作る

本文では、`rtt_measure.py` を3回（`--label lan`・`ts_home`・`ts_outside`、300 回ずつ、0.2 秒おき）実行して `rtt.csv` に追記します。ここでは `synth_rtt.py` が、同じ形の `rtt_synth.csv` を一度に作ります。真の中央値（とても長い列で求めた値）は `rtt_synth_truth.json` に書き、確かめだけに使います。

In [ ]:
run("synth_rtt.py --out rtt_synth.csv --count 300 --seed 1")
d = pd.read_csv("rtt_synth.csv")
truth = json.load(open("rtt_synth_truth.json"))
print(d.head())
# 確かめ（手順：rtt.csv に3つのラベルが300行ずつ入る）
ok(list(d.columns) == ["label", "when", "seq", "rtt_ms"], "列は rtt_measure.py と同じ（label, when, seq, rtt_ms）")
ok(d.groupby("label").size().to_dict() == {"lan": 300, "ts_home": 300, "ts_outside": 300}, "3つのラベルが300行ずつ")

## 2. 経路ごとに比べる（`rtt_compare.py`）

ブートストラップを経路ごとに 3000 回×2通り行うので、10 秒ほどかかります。図は `rtt.png` に保存されます。

In [ ]:
out = run("rtt_compare.py rtt_synth.csv")
display(Image(filename="rtt.png", width=900))

In [ ]:
# 確かめ（手順：中央値・95%点・欠損率、2通りの区間、中央値の差が表示され、rtt.png ができる。ts_home − lan の差は小さく、ts_outside は大きい）
stat = {}
for lb in ["lan", "ts_home", "ts_outside"]:
    m = re.search(lb + r"\s+(\d+)/(\d+)\s+([\d.]+)\s+\(([\d.]+)〜([\d.]+)\)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+(-?[\d.]+)", out)
    m2 = re.search(lb + r"[^\n]*\n\s+中央値の95%区間：独立とみなす ([\d.]+)〜([\d.]+) ms、ブロック ([\d.]+)〜([\d.]+) ms", out)
    stat[lb] = dict(med=float(m.group(6)), q95=float(m.group(7)), rho1=float(m.group(9)),
                    iid=(float(m2.group(1)), float(m2.group(2))), blk=(float(m2.group(3)), float(m2.group(4))))
    s = stat[lb]
    print(lb, s)
ok(os.path.exists("rtt.png"), "rtt.png ができた")
diff_home = num(r"ts_home − lan：([+-][\d.]+) ms", out)
diff_out = num(r"ts_outside − lan：([+-][\d.]+) ms", out)
ok(abs(diff_home) < 3, f"ts_home − lan = {diff_home:+.2f} ms（数 ms 以内。合成データの真の差 {truth['ts_home'] - truth['lan']:+.2f} ms）")
ok(diff_out > 20, f"ts_outside − lan = {diff_out:+.2f} ms（ts_outside は大きい）")
for lb, s in stat.items():
    ok(s["q95"] > 1.5 * s["med"], f"{lb}：95%点 {s['q95']} ms は中央値 {s['med']} ms よりずっと大きい（裾が重い）")
    w_iid, w_blk = s["iid"][1] - s["iid"][0], s["blk"][1] - s["blk"][0]
    ok(w_blk > w_iid, f"{lb}：ブロックの区間の幅 {w_blk:.2f} ms は、独立とみなした区間 {w_iid:.2f} ms より広い（ρ(1) = {s['rho1']}）")

本文の手順の「確かめること」には、「ρ(1) が大きい経路ほど、2つの区間の幅の違いが大きい」とも書きました。上の表で、ρ(1) と、幅の比（ブロック ÷ 独立）を並べてみます。

In [ ]:
for lb, s in stat.items():
    ratio = (s["blk"][1] - s["blk"][0]) / (s["iid"][1] - s["iid"][0])
    x = d[d["label"] == lb]["rtt_ms"].dropna().to_numpy()
    rs = pd.Series(x[:-1]).corr(pd.Series(x[1:]), method="spearman")
    print(f"{lb:11s} ρ(1) = {s['rho1']:.2f}（順位で求めると {rs:.2f}）  幅の比 {ratio:.2f}")

この合成データでは、ρ(1) の順と幅の比の順はそろいません。`rtt_compare.py` の ρ(1) は、値そのもの（ピアソン）の相関なので、混雑のときの数十〜数百 ms の少数の値に大きく引っぱられます。一方、中央値の区間の幅を決めるのは、ふだんの値（分布の真ん中）が続けて似ているかどうかです。次の 3 節で、たくさんくり返して確かめます。

## 3. 被覆率の実験：独立とみなした区間は狭すぎるか（ノートブックで追加）

答えのわかった合成データなので、同じ経路の 300 回の測定を何度も作り直して、「95% 区間が真の中央値を含んだ割合」（被覆率）を数えられます。正しい 95% 区間なら約 95% になるはずです。`rtt_compare.py` の `iid_boot`・`block_boot` をそのまま使い、時間を短くするためにブートストラップの回数 `NB` を 3000 から 300 に減らします（区間の端の値が少しぶれるだけで、被覆率の傾向は変わりません）。`R = 200` 回で約5秒です。

In [ ]:
import rtt_compare as rc
import synth_rtt as sy
rc.NB = 300            # 本文は 3000。被覆率の実験では回数を減らす
R = 200                # 作り直す回数（増やすと被覆率の誤差が小さくなる。R = 1000 で約30秒）
rng = np.random.default_rng(5)
cov = {}
for lb in ["lan", "ts_outside"]:
    hit_i = hit_b = 0
    ratio, r1, rs = [], [], []
    for _ in range(R):
        x = sy.simulate(lb, 300, rng)
        x = x[~np.isnan(x)]
        a, b = rc.iid_boot(x), rc.block_boot(x)
        lo, hi = np.percentile(a, [2.5, 97.5])
        lo2, hi2 = np.percentile(b, [2.5, 97.5])
        hit_i += lo <= truth[lb] <= hi
        hit_b += lo2 <= truth[lb] <= hi2
        ratio.append((hi2 - lo2) / (hi - lo))
        r1.append(np.corrcoef(x[:-1], x[1:])[0, 1])
        rs.append(pd.Series(x[:-1]).corr(pd.Series(x[1:]), method="spearman"))
    cov[lb] = dict(iid=hit_i / R, blk=hit_b / R, c_r1=np.corrcoef(r1, ratio)[0, 1], c_rs=np.corrcoef(rs, ratio)[0, 1],
                   ratio=np.median(ratio))
    c = cov[lb]
    print(f"{lb:11s} 被覆率：独立とみなす {c['iid']:.0%}、ブロック {c['blk']:.0%}（幅の比の中央値 {c['ratio']:.2f}）"
          f"  幅の比との相関：ρ(1) {c['c_r1']:+.2f}、順位の ρ(1) {c['c_rs']:+.2f}")

In [ ]:
# 確かめ（II-2-3：独立とみなしたブートストラップは区間が狭すぎる。ブロックのほうが 95% に近い）
for lb, c in cov.items():
    ok(c["iid"] < 0.80, f"{lb}：独立とみなした 95% 区間の被覆率は {c['iid']:.0%}（95% よりずっと低い＝狭すぎる）")
    ok(c["blk"] > c["iid"] + 0.1, f"{lb}：ブロック・ブートストラップの被覆率 {c['blk']:.0%} のほうが 95% に近い")
ok(all(abs(c["c_r1"]) < abs(c["c_rs"]) for c in cov.values()),
   "幅の比を説明するのは、値そのものの ρ(1) より順位の ρ(1)（大きい値に引っぱられない）")

- 独立とみなした区間は、真の中央値を 95% よりずっと少ない割合でしか含みません。0.2 秒おきの ping は、となりどうしが似ているので、300 回は「独立な 300 個」ぶんの情報を持っていないからです。
- ブロック・ブートストラップ（長さ 20）でも、この合成データでは 95% に届きませんでした。混雑が続く長さや、ふだんの値の似ている長さが、ブロックの長さや測定の長さ（300 回 = 1 分）に比べて無視できないためです。本文のとおり、どちらの区間も「その日・その時間帯の 1 回の測定」についてのもので、ほんとうのくり返しは、日を変えて測ることです。
- 幅の比を説明するのは、値そのものの ρ(1) より、順位で求めた ρ(1) でした。裾の重いデータで「となりどうしが似ているか」を見るなら、順位の相関や、log RTT の相関のほうが当てになります。

## 4. `rtt_measure.py`：ping の表示から RTT を読む

`rtt_measure.py` は、`ping -c 300 -i 0.2 ホスト` の表示の各行から、正規表現 `icmp_seq=(\d+).*?time=([\d.]+)\s*ms` で番号と時間を読み、届かなかった番号を空欄にします。macOS の ping は `icmp_seq` を 0 から、Linux（iputils）の ping は 1 から数えます。`rtt_measure.py` は Mac 用なので 0 から数える（`first = 0`）と決めています。次のセルで、両方の表示の見本を読ませてみます。

In [ ]:
LINE = re.compile(r"icmp_seq=(\d+).*?time=([\d.]+)\s*ms")    # rtt_measure.py と同じ
mac = """PING labpi.local (192.168.1.50): 56 data bytes
64 bytes from 192.168.1.50: icmp_seq=0 ttl=64 time=4.012 ms
Request timeout for icmp_seq 1
64 bytes from 192.168.1.50: icmp_seq=2 ttl=64 time=3.877 ms"""
linux = """PING labpi.local (192.168.1.50) 56(84) bytes of data.
64 bytes from 192.168.1.50: icmp_seq=1 ttl=64 time=4.01 ms
64 bytes from 192.168.1.50: icmp_seq=3 ttl=64 time=3.88 ms"""
for name, text, count in [("macOS", mac, 3), ("Linux", linux, 3)]:
    got = {int(m.group(1)): float(m.group(2)) for m in map(LINE.search, text.splitlines()) if m}
    rows = [(s, got.get(s, "")) for s in range(0, count)]     # rtt_measure.py の first = 0 と同じ
    print(f"{name}: 読めた {got} → CSV の (seq, rtt_ms) = {rows}")
got_mac = {int(m.group(1)): float(m.group(2)) for m in map(LINE.search, mac.splitlines()) if m}
ok(got_mac == {0: 4.012, 2: 3.877}, "macOS の表示から、届いた番号と時間を読み、Request timeout の行は読まない（→ seq 1 は空欄）")
print("※ Linux の ping の表示をそのまま読むと、番号が1つずれる（seq 0 が空欄になり、最後の1回が数えられない）。Linux で使うなら first = 1 にする")

Mac で実行しているときだけ、自分自身（127.0.0.1）に 10 回 ping して、`rtt_measure.py` を本物の ping で動かします（2 秒ほど。Colab は Linux で、ping が入っていないこともあるので、とばします）。

In [ ]:
if sys.platform == "darwin" and shutil.which("ping"):
    if os.path.exists("rtt_local.csv"):
        os.remove("rtt_local.csv")
    o = run("rtt_measure.py --host 127.0.0.1 --label loopback --count 10 --interval 0.2 --out rtt_local.csv")
    loc = pd.read_csv("rtt_local.csv")
    print(loc)
    ok(len(loc) == 10 and loc["rtt_ms"].notna().all() and loc["rtt_ms"].max() < 5, "127.0.0.1 には 10 回とも届き、RTT は 1 ms 前後以下")
else:
    print("Mac ではないので、本物の ping はとばします")

## おわりに

- RTT は裾が重いので、平均ではなく中央値・95%点・経験分布関数で比べます。欠損率は Wilson 区間で示します。
- 続けて打った ping を独立とみなすと、区間は狭すぎます（この合成データでは、95% のはずの区間が真の値を含んだのは4〜6割ほど）。ブロック・ブートストラップはそれを和らげますが、別の日のばらつきまでは含みません。
- となりどうしの似かたを ρ(1)（値そのものの相関）で見ると、少数の大きな値に引っぱられます。順位の相関も合わせて見ると安全です。
- このノートブックに入れていないもの：Tailscale の導入と設定、実際の経路（LAN・tailnet・外出先）での測定。これらは教科書の手順のとおりに Mac と Pi で行います。